# Subreddit baseline

Use subreddit comment counts to predict the five Big Five scores.
Run the cells in order.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from scipy.stats import pearsonr
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

DATA = Path("../data/preprocessed")  # From the project's src folder
OUTPUT = Path("../data/subreddit_baseline")
TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
ALPHAS = [0.01, 0.1, 1, 10, 100]

## 1. Get the counts for each user

Each user gets a dictionary like {"askreddit": 12, "books": 3}.
The model learns the subreddit columns and IDF from its training users.
Subreddits missing from training are ignored at test time.

In [ ]:
users = pd.read_csv(DATA / "main/users.csv").set_index("author")
outer = pd.read_csv(DATA / "outer_folds.csv").set_index("author")
inner = pd.read_csv(DATA / "inner_folds.csv")
assert users.index.is_unique and outer.index.is_unique
assert set(users.index) == set(outer.index)

counts = pd.read_csv(DATA / "main/subreddit_counts.csv")
subreddit_counts = {}
for author, rows in counts.groupby("author"):
    subreddit_counts[author] = dict(zip(rows["subreddit"], rows["comment_count"]))

data = users[TRAITS].join(outer)
data["counts"] = data.index.map(subreddit_counts)
assert data["counts"].notna().all()
print("Users:", len(data))

## 2. Train and predict

For each outer fold, pick alpha separately for each trait by inner validation
RMSE. The pipeline fits the subreddit columns and TF-IDF again each time it trains. We also predict the
outer training mean as a reference.

In [ ]:
predictions = []

for fold in sorted(data["outer_fold"].unique()):
    train = data[data["outer_fold"] != fold]
    test = data[data["outer_fold"] == fold]

    inner_fold = inner[inner["outer_fold"] == fold].set_index("author")["inner_fold"]
    assert inner_fold.index.is_unique and set(inner_fold.index) == set(train.index)
    inner_fold = inner_fold.reindex(train.index).to_numpy()
    cv = []
    for val_fold in sorted(set(inner_fold)):
        train_idx = (inner_fold != val_fold).nonzero()[0]
        val_idx = (inner_fold == val_fold).nonzero()[0]
        cv.append((train_idx, val_idx))

    # Ridge can choose a different alpha for each trait. The TF-IDF input
    # is the same for all traits, so cache it for the searches in this fold.
    with TemporaryDirectory() as cache:
        model = make_pipeline(
            DictVectorizer(),
            TfidfTransformer(),
            Ridge(solver="lsqr"),
            memory=cache,
        )

        for trait in TRAITS:
            search = GridSearchCV(model, {"ridge__alpha": ALPHAS}, cv=cv,
                                  scoring="neg_root_mean_squared_error")
            search.fit(train["counts"], train[trait])
            predictions.append(pd.DataFrame({
                "author": test.index,
                "outer_fold": fold,
                "trait": trait,
                "true": test[trait].to_numpy(),
                "subreddit_prediction": search.predict(test["counts"]),
                "mean_prediction": train[trait].mean(),
                "alpha": search.best_params_["ridge__alpha"],
            }))
    print("Finished fold", fold)

## 3. Check the results

Each user has one test prediction per trait. Compute the metrics using those
predictions. Pearson correlation is left blank if the predictions are constant.

In [ ]:
results = pd.concat(predictions, ignore_index=True)
assert len(results) == len(users) * len(TRAITS)
assert not results.duplicated(["author", "trait"]).any()

metrics = []
for trait, rows in results.groupby("trait"):
    actual = rows["true"]
    for model in ["subreddit", "mean"]:
        predicted = rows[f"{model}_prediction"]
        correlation = float("nan")
        if predicted.nunique() > 1 and actual.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted),
        })

metrics = pd.DataFrame(metrics)
OUTPUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
display(metrics.round(3))